# Notebook 02 - Q1: Phân tích Chuyên sâu Câu hỏi Q1 (VeDDRA & Xu hướng theo quý)
**Đề tài:** Phân tích Sự cố Bất lợi Thuốc Thú y (OpenFDA Animal & Veterinary Adverse Events) giai đoạn 2025 và Đối chiếu Xu hướng 2026 (Q1)

**Môn học:** CSC17104 – Lập trình cho Khoa học Dữ liệu  
**Nhóm thực hiện:** Global Ban  
**Thành viên phụ trách:** Đinh Công Khang  
**Phản biện chéo:** Huỳnh Tuấn Kiệt  
**Liên kết Master Hub:** [Notebook 02: Master Hub (Mục 3.3 Question Formulation & D1–D6)](02_analysis.ipynb)


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập đường dẫn gốc dự án
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_processed
from src.preprocessing import log_row_count_change
from src.visualization import set_project_plot_style
set_project_plot_style()

In [2]:
# Tải dữ liệu đã qua tiền xử lý chuẩn hóa từ data/processed/
data = load_processed(PROJECT_ROOT / 'data' / 'processed')
reports = data['reports']
reactions = data['reactions']
drugs = data['drugs']
outcomes = data['outcomes']

print(f"[Data Loaded] Reports: {reports.shape}, Reactions: {reactions.shape}, Drugs: {drugs.shape}, Outcomes: {outcomes.shape}")

[Data Loaded] Reports: (94412, 33), Reactions: (270244, 8), Drugs: (174123, 31), Outcomes: (85143, 5)


---
## Câu hỏi Q1: Phản ứng VeDDRA hàng đầu & Xu hướng biến động theo quý

### Phát biểu câu hỏi và Động lực
- **Phát biểu câu hỏi:**  
  > **"Những phản ứng bất lợi (VeDDRA terms) nào chiếm tỷ trọng cao nhất ở Chó, Mèo và các loài gia súc/vật nuôi khác trong năm 2025, và xu hướng theo quý (kèm dấu hiệu mùa vụ trong phạm vi 1 năm) của các phản ứng hàng đầu này biến động như thế nào cho đến nửa đầu năm 2026 (Q1)?"**
- **Người quan tâm & Quyết định hỗ trợ (Target Users & Decisions):**
  - **Bác sĩ thú y lâm sàng:** Cần nắm bắt mô hình triệu chứng phổ biến nhất theo từng loài cụ thể để chẩn đoán phân biệt và nhận diện sớm các phản ứng bất lợi sau điều trị theo từng mùa trong năm.
  - **Cơ quan quản lý (FDA CVM) & Nhà sản xuất thuốc:** Cần giám sát tín hiệu an toàn thời gian thực để phát hiện các đợt bùng phát bất thường (outbreaks) hoặc sự gia tăng đột biến của phản ứng phụ theo quý.
- **Tiêu chí độ sâu áp dụng (Mục 3.3):**
  - `D1` (Combining data): Nối bảng báo cáo cha `reports` với bảng triệu chứng lâm sàng con `reactions` theo khóa `unique_aer_id_number` (kiểm tra số dòng trước/sau và tỷ lệ khớp).
  - `D2` (Reshaping & time): Resample dữ liệu theo quý (2025 Q1 — 2026 Q1), xoay bảng (pivot table) tần suất phản ứng theo thời gian và tính tỷ lệ biến động qua từng kỳ.
  - `D5` (Quantifying uncertainty): Ước lượng khoảng tin cậy 95% Bootstrap cho tỷ lệ phản ứng theo quý và thực hiện kiểm định $\chi^2$ về tính đồng nhất phân bố mùa vụ.

### 1. Tiền xử lý dữ liệu cho Q1

1. **Bước 1.1 — Lọc quần thể thú y mục tiêu:**
   - Dữ liệu openFDA Adverse Events bao gồm cả một số lượng nhỏ ca báo cáo trên người (`species == "Human"` — thường do người vô tình tiếp xúc với thuốc thú y). Do đề tài tập trung vào sức khỏe vật nuôi, nhóm loại bỏ toàn bộ các bản ghi `Human`.
2. **Bước 1.2 — Kết nối đa bảng theo chiều sâu (Tiêu chí D1 — Combining data):**
   - Nối bảng báo cáo cha `reports` với bảng triệu chứng con `reactions` thông qua khóa chính/ngoại `unique_aer_id_number` (Inner Join).
   - Kiểm tra và xác nhận tính toàn vẹn tham chiếu (Zero orphan keys): xác định số lượng và tỷ lệ báo cáo không có thông tin triệu chứng (thường là nhóm báo cáo *Product Defect*).
3. **Bước 1.3 — Giới hạn phạm vi thời gian nghiên cứu (5 quý liên tiếp):**
   - Lọc các bản ghi thuộc 5 phân vùng quý từ `2025q1` đến `2026q1`.
   - Làm sạch trường thuật ngữ phản ứng `veddra_term_name`: loại bỏ các chuỗi rỗng và chuỗi giữ chỗ (`NaN`, `Unknown`, `None`, `<NA>`).
4. **Bước 1.4 — Phân loại nhóm loài & Khử trùng lặp cấp (Ca bệnh, Triệu chứng, Quý) (Tiêu chí D2):**
   - Ánh xạ loài thành 3 nhóm lớn: **Chó (`Dog`)**, **Mèo (`Cat`)**, và **Các loài khác (`Other`)**.
   - Khử trùng lặp theo tổ hợp `(unique_aer_id_number, veddra_term_name, quarter, species_group)` để đảm bảo mỗi ca bệnh chỉ đóng góp tối đa 1 lần cho cùng một thuật ngữ VeDDRA trong một quý.


In [5]:
# ===== Tiền xử lý dữ liệu phục vụ câu hỏi Q1 =====
# Bước 1.1: Lọc quần thể thú y (loại bỏ Human)
before_rep = len(reports)
rep_animal = reports.loc[~reports["species"].fillna("").str.lower().eq("human")].copy()
log_row_count_change("Q1_filter_non_human", before_rep, len(rep_animal), 
                     "Loại bỏ các ca báo cáo trên người (Human) để chỉ tập trung vào thú y")

# Bước 1.2: Kết nối bảng cha reports với bảng con reactions (D1)
before_rea = len(reactions)
KEY = "unique_aer_id_number"
rep_cols = [KEY, "species", "primary_reporter", "serious_ae"]
rep_cols = [c for c in rep_cols if c in rep_animal.columns]

q1_merged = reactions.merge(rep_animal[rep_cols], on=KEY, how="inner")
log_row_count_change("Q1_merge_reactions_reports", before_rea, len(q1_merged), 
                     "Inner join reactions x reports theo khóa unique_aer_id_number")

# Kiểm tra tính toàn vẹn của phép kết nối: số lượng báo cáo không khớp
unmatched_reports = rep_animal[KEY].nunique() - q1_merged[KEY].nunique()
print(f"Số ca báo cáo không có phản ứng nào trong reactions.csv: {unmatched_reports:,} ({unmatched_reports/rep_animal[KEY].nunique()*100:.2f}%)")

# Bước 1.3: Lọc phạm vi 5 quý (2025 Q1 — 2026 Q1)
QUARTERS = ["2025q1", "2025q2", "2025q3", "2025q4", "2026q1"]
q1_merged["quarter"] = q1_merged["quarter"].astype(str).str.strip().str.lower()
before_q = len(q1_merged)
q1_merged = q1_merged[q1_merged["quarter"].isin(QUARTERS)].copy()
log_row_count_change("Q1_filter_quarters", before_q, len(q1_merged), 
                     "Giới hạn dữ liệu trong 5 quý nghiên cứu (2025 Q1 - 2026 Q1)")

# Làm sạch thuật ngữ VeDDRA (loại bỏ chuỗi rỗng / placeholder ẩn)
MISSING_TOKENS = {"", "nan", "none", "unknown", "<na>", "null"}
q1_merged["veddra_term_name"] = q1_merged["veddra_term_name"].astype(str).str.strip()
before_clean = len(q1_merged)
q1_merged = q1_merged[~q1_merged["veddra_term_name"].str.lower().isin(MISSING_TOKENS)].copy()
log_row_count_change("Q1_clean_veddra_terms", before_clean, len(q1_merged), 
                     "Loại bỏ thuật ngữ VeDDRA khuyết hoặc rỗng")

# Bước 1.4: Phân nhóm loài và khử trùng lặp (1 term / 1 ca bệnh trong quý)
def map_species_group(s):
    sl = str(s).strip().lower()
    if sl == "dog":
        return "Dog"
    elif sl == "cat":
        return "Cat"
    return "Other"

q1_merged["species_group"] = q1_merged["species"].map(map_species_group)

before_dedup = len(q1_merged)
q1_clean = q1_merged.drop_duplicates(
    subset=[KEY, "veddra_term_name", "quarter", "species_group"]
).copy()
log_row_count_change("Q1_dedup_report_term", before_dedup, len(q1_clean), 
                     "Mỗi ca bệnh chỉ tính tối đa 1 lần cho mỗi thuật ngữ VeDDRA trong quý")

print(f"\n[Q1 Tiền xử lý hoàn tất] Dữ liệu sạch cho Q1: {len(q1_clean):,} dòng ({q1_clean[KEY].nunique():,} ca báo cáo duy nhất).")


[Q1_filter_non_human] Rows: 94,412 -> 93,812 (Change: -600 | -0.64%) | Reason: Loại bỏ các ca báo cáo trên người (Human) để chỉ tập trung vào thú y
[Q1_merge_reactions_reports] Rows: 270,244 -> 268,426 (Change: -1,818 | -0.67%) | Reason: Inner join reactions x reports theo khóa unique_aer_id_number
Số ca báo cáo không có phản ứng nào trong reactions.csv: 66 (0.07%)
[Q1_filter_quarters] Rows: 268,426 -> 268,426 (Change: +0 | +0.00%) | Reason: Giới hạn dữ liệu trong 5 quý nghiên cứu (2025 Q1 - 2026 Q1)
[Q1_clean_veddra_terms] Rows: 268,426 -> 268,426 (Change: +0 | +0.00%) | Reason: Loại bỏ thuật ngữ VeDDRA khuyết hoặc rỗng
[Q1_dedup_report_term] Rows: 268,426 -> 267,848 (Change: -578 | -0.22%) | Reason: Mỗi ca bệnh chỉ tính tối đa 1 lần cho mỗi thuật ngữ VeDDRA trong quý

[Q1 Tiền xử lý hoàn tất] Dữ liệu sạch cho Q1: 267,848 dòng (93,746 ca báo cáo duy nhất).


### 2. Phương pháp phân tích

*(Giải thích phương pháp được chọn và lý do phương pháp này phù hợp để giải quyết câu hỏi)*:
- **Phương pháp 1:** Xếp hạng Top 10 phản ứng VeDDRA có tần suất xuất hiện cao nhất, phân tầng độc lập cho Chó, Mèo và các loài khác trong năm 2025.
- **Phương pháp 2:** Tính toán tỷ lệ phần trăm của từng phản ứng trong tổng số báo cáo của từng quý từ 2025 Q1 đến 2026 Q1.
- **Phương pháp 3:** Tính khoảng tin cậy 95% Bootstrap cho tỷ lệ phản ứng hàng đầu theo từng quý và kiểm định $\chi^2$ kiểm tra xem xu hướng có thay đổi có ý nghĩa thống kê giữa các quý không.

In [ ]:
# TODO [Khang - M1]: Triển khai phân tích dữ liệu Q1 (Xếp hạng VeDDRA, Resampling theo quý, Bootstrap CI)

### 3. Kết quả và Trực quan hóa

- **Biểu đồ 1:** Barplot phân bố Top 10 thuật ngữ VeDDRA phổ biến nhất theo từng loài (Chó vs Mèo vs Khác).
- **Biểu đồ 2:** Lineplot biến thiên chuỗi thời gian 5 quý (2025 Q1 — 2026 Q1) của các phản ứng chủ đạo kèm dải tin cậy 95% (shaded confidence intervals).

In [ ]:
# TODO [Khang - M1]: Vẽ tối thiểu 2 biểu đồ chuẩn mực cho Q1

### 4. Trả lời câu hỏi và Thảo luận

- **Câu trả lời tường minh (Explicit Answer with specific numbers):**  
  *(Nêu rõ các con số thống kê cụ thể: Tên phản ứng VeDDRA phổ biến nhất, tỷ lệ % chiếm dụng, độ tăng giảm % theo quý, giá trị p-value của kiểm định thống kê)*.
- **Ý nghĩa thực tiễn & Bất ngờ (Practical Meaning & Surprises):**  
  *(Phát hiện có gì bất ngờ về triệu chứng giữa chó và mèo? Có dấu hiệu mùa vụ rõ rệt ở phản ứng nào không — ví dụ bệnh ngoài da hoặc sốc thuốc mùa nóng?)*
- **Hạn chế của phân tích (Limitations):**  
  *(Ảnh hưởng của cơ chế tự nguyện báo cáo, sự sai khác trong thói quen ghi nhận thuật ngữ VeDDRA của các phòng khám thú y)*.

---
## Điều hướng liên kết
- **Trước:** [Master Hub](02_analysis.ipynb)
- **Master Hub:** [Notebook 02: Tổng quan & Ma trận D1–D6](02_analysis.ipynb)
- **Tiếp theo:** [Câu hỏi Q2](02_q2_analysis.ipynb)
